In [2]:
import pandas as pd
nykaa_df = pd.read_csv('nykaa_cleaned.csv')
tira_df = pd.read_csv('tira_cleaned.csv')
purpulle_df = pd.read_csv('purplle_cleaned.csv')
combined_df = pd.concat([nykaa_df, tira_df, purpulle_df], ignore_index=True)

print("Combined DataFrame head:")
print(combined_df.head())
print("\nCombined DataFrame shape:", combined_df.shape)

Combined DataFrame head:
   Campaign_ID Campaign_Type        Target_Audience  Duration  \
0  NY-CMP-1000  Social Media       College Students      21.0   
1  NY-CMP-1001      Paid Ads  Tier 2 City Customers      18.0   
2  NY-CMP-1002    Influencer                  Youth      23.0   
3  NY-CMP-1003         Email          Working Women      18.0   
4  NY-CMP-1004      Paid Ads       College Students      10.0   

                   Channel_Used  Impressions  Clicks   Leads  Conversions  \
0             WhatsApp, YouTube      57804.0  6156.0  3616.0       2355.0   
1                       YouTube      91801.0  3321.0  1971.0       1357.0   
2     WhatsApp, Google, YouTube      15536.0  2182.0   952.0        755.0   
3  YouTube, Facebook, Instagram      88114.0  8413.0  2231.0        947.0   
4           Facebook, Instagram      96871.0  3743.0  2060.0       1258.0   

     Revenue  Acquisition_Cost      ROI Language  Engagement_Score  \
0  1867515.0            65.048     6.14    Hindi   

In [3]:
combined_df.to_csv('combined_df.csv', index=False)

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

combined_df['Profit'] = combined_df['Revenue'] - combined_df['Acquisition_Cost']

combined_df['Profit_Loss_Category'] = combined_df['Profit'].apply(lambda x: 'Profit' if x > 0 else 'Loss')

print("Combined DataFrame head with new columns:")
display(combined_df.head())
print("\nDistribution of Profit/Loss Category:")
display(combined_df['Profit_Loss_Category'].value_counts())

Combined DataFrame head with new columns:


,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Conversion_Rate,Profit,Profit_Loss_Category
0,NY-CMP-1000,Social Media,College Students,21.0,"WhatsApp, YouTube",57804.0,6156.0,3616.0,2355.0,1867515.0,65.048,6.14,Hindi,20.98,College Students,0.382554,1867449.952,Profit
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18.0,YouTube,91801.0,3321.0,1971.0,1357.0,1046247.0,180.830,3.26,Hindi,7.24,College Students,0.408612,1046066.170,Profit
2,NY-CMP-1002,Influencer,Youth,23.0,"WhatsApp, Google, YouTube",15536.0,2182.0,952.0,755.0,197055.0,90.600,2175.00,English,25.03,College Students,0.346013,196964.400,Profit
3,NY-CMP-1003,Email,Working Women,18.0,"YouTube, Facebook, Instagram",88114.0,8413.0,2231.0,947.0,376906.0,249.070,0.60,Hindi,13.15,Unknown,0.112564,376656.930,Profit
4,NY-CMP-1004,Paid Ads,College Students,10.0,"Facebook, Instagram",96871.0,3743.0,2060.0,1258.0,518296.0,228.600,0.80,Hindi,7.29,Tier 2 City Customers,0.336094,518067.400,Profit



Distribution of Profit/Loss Category:


Profit_Loss_Category
Profit    161238
Loss          10
Name: count, dtype: int64

In [5]:
# Identify categorical and numerical columns
categorical_cols = combined_df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = combined_df.select_dtypes(include=['number']).columns.tolist()


feature_exclude= ['Campaign_ID', 'Revenue', 'Profit', 'Profit_Loss_Category']


categorical_cols = [col for col in categorical_cols if col not in feature_exclude]
numerical_cols = [col for col in numerical_cols if col not in feature_exclude]

combined_df_encoded = pd.get_dummies(combined_df, columns=categorical_cols, drop_first=True)

# Define features (X) and targets (y)
X = combined_df_encoded.drop(columns=[col for col in feature_exclude if col in combined_df_encoded.columns], errors='ignore')
y_revenue = combined_df_encoded['Revenue']
y_profit = combined_df_encoded['Profit']
y_profit_loss = combined_df_encoded['Profit_Loss_Category']

print("Shape of feature matrix X:", X.shape)
print("Shape of Revenue target y:", y_revenue.shape)
print("Shape of Profit target y:", y_profit.shape)
print("Shape of Profit/Loss target y:", y_profit_loss.shape)

Shape of feature matrix X: (161248, 184)
Shape of Revenue target y: (161248,)
Shape of Profit target y: (161248,)
Shape of Profit/Loss target y: (161248,)


C:\Users\R.K.DEEPIKA\AppData\Local\Temp\ipykernel_14136\1668781523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = combined_df.select_dtypes(include=['object']).columns.tolist()


In [6]:
# Split data for Revenue prediction (regression)
X_train_rev, X_test_rev, y_train_rev, y_test_rev = train_test_split(X, y_revenue, test_size=0.2, random_state=42)

# Split data for Profit prediction (regression)
X_train_prof, X_test_prof, y_train_prof, y_test_prof = train_test_split(X, y_profit, test_size=0.2, random_state=42)

# Split data for Profit/Loss prediction (classification) with stratification
X_train_pl, X_test_pl, y_train_pl, y_test_pl = train_test_split(X, y_profit_loss, test_size=0.2, random_state=42, stratify=y_profit_loss)

print("Training set size for Revenue:", X_train_rev.shape)
print("Test set size for Revenue:", X_test_rev.shape)
print("Training set size for Profit:", X_train_prof.shape)
print("Test set size for Profit:", X_test_prof.shape)
print("Training set size for Profit/Loss classification:", X_train_pl.shape)
print("Test set size for Profit/Loss classification:", X_test_pl.shape)

Training set size for Revenue: (128998, 184)
Test set size for Revenue: (32250, 184)
Training set size for Profit: (128998, 184)
Test set size for Profit: (32250, 184)
Training set size for Profit/Loss classification: (128998, 184)
Test set size for Profit/Loss classification: (32250, 184)


In [ ]:
rf_revenue_model = RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1) 
rf_revenue_model.fit(X_train_rev, y_train_rev)
y_pred_rev = rf_revenue_model.predict(X_test_rev)

In [8]:
mae_revenue = mean_absolute_error(y_test_rev, y_pred_rev)
print(f"Revenue Prediction MAE: {mae_revenue:.2f}")


Revenue Prediction MAE: 50950.42


In [9]:
r2_revenue = r2_score(y_test_rev, y_pred_rev)
print(f"Revenue Prediction R2 Score: {r2_revenue:.2f}")

Revenue Prediction R2 Score: 0.92


In [10]:
rf_profit_model = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1) # Use all available cores
rf_profit_model.fit(X_train_prof, y_train_prof)

y_pred_prof = rf_profit_model.predict(X_test_prof)

mae_profit = mean_absolute_error(y_test_prof, y_pred_prof)
r2_profit = r2_score(y_test_prof, y_pred_prof)

print(f"Profit Prediction MAE: {mae_profit:.2f}")
print(f"Profit Prediction R2 Score: {r2_profit:.2f}")

Profit Prediction MAE: 47634.35
Profit Prediction R2 Score: 0.93


In [12]:
rf_profit_loss_model = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1) # Use all available cores
rf_profit_loss_model.fit(X_train_pl, y_train_pl)

y_pred_pl = rf_profit_loss_model.predict(X_test_pl)

accuracy_pl = accuracy_score(y_test_pl, y_pred_pl)
report_pl = classification_report(y_test_pl, y_pred_pl)

print(f"Profit/Loss Classification Accuracy: {accuracy_pl:.2f}")
print("\nProfit/Loss Classification Report:\n", report_pl)

Profit/Loss Classification Accuracy: 1.00

Profit/Loss Classification Report:
               precision    recall  f1-score   support

        Loss       0.00      0.00      0.00         2
      Profit       1.00      1.00      1.00     32248

    accuracy                           1.00     32250
   macro avg       0.50      0.50      0.50     32250
weighted avg       1.00      1.00      1.00     32250



c:\Users\R.K.DEEPIKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\R.K.DEEPIKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\R.K.DEEPIKA\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio